## Comparison vs paper

- The paper's **77.11%** cross-validation accuracy (KNN, **no** duplication, two segments) is a reasonable **pixel-model target** when your dataset is similarly small and noisy.
- Reported **96%** with duplication inflates performance; **OOF AUC and balanced sensitivity/specificity** on **non-duplicated** rows are more trustworthy.
- With **real** OCT `.tif` data, expect to tune `IMG_SIZE`, augmentation, `EPOCHS_CNN`, and consider **patient-level splits** (same eye / patient in one fold) if you have IDs in a CSV column `patient_id`.

In [ ]:
# Additional ML models on robust image features (standalone cell)
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)


def _compute_metrics_local(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    sens = recall_score(y_true, y_pred, zero_division=0)
    spec = recall_score(1 - y_true, 1 - y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = float("nan")

    # Likelihood ratios used in paper
    lr_pos = sens / (1 - spec + 1e-12)
    lr_neg = (1 - sens) / (spec + 1e-12)

    return {
        "accuracy": acc,
        "sensitivity": sens,
        "specificity": spec,
        "precision": prec,
        "f1": f1,
        "auc": auc,
        "lr_pos": lr_pos,
        "lr_neg": lr_neg,
        "y_true": y_true,
        "y_score": y_score,
        "y_pred": y_pred,
    }


def _plot_cm_local(y_true, y_pred, title="Confusion matrix"):
    import matplotlib.pyplot as plt
    import seaborn as sns

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def _safe_collect_samples(data_root: Path) -> pd.DataFrame:
    """Collect TIFF samples from dataset_index.csv, class folders, or flat files (heuristic labels)."""
    label_pos = {"positive", "thinning", "1", "yes", "true", "scm"}
    label_neg = {"negative", "no_thinning", "normal", "0", "no", "false"}
    rows = []

    csv_path = data_root / "dataset_index.csv"
    if csv_path.is_file():
        tmp = pd.read_csv(csv_path)
        for _, r in tmp.iterrows():
            p = Path(r["path"])
            if not p.is_file():
                p = data_root / r["path"]
            lab = r["label"]
            if isinstance(lab, str):
                lab = 1 if lab.strip().lower() in label_pos else 0
            rows.append({"path": p.resolve(), "label": int(lab)})
        return pd.DataFrame(rows)

    # 1) Class-folder mode
    if data_root.exists():
        for sub in data_root.iterdir():
            if not sub.is_dir():
                continue
            name = sub.name.lower()
            if name in label_pos:
                label = 1
            elif name in label_neg:
                label = 0
            else:
                continue

            for f in sub.rglob("*"):
                if f.suffix.lower() in {".tif", ".tiff"}:
                    rows.append({"path": f.resolve(), "label": label})

    if rows:
        return pd.DataFrame(rows)

    # 2) Flat-file mode in DATA_ROOT (no dataset_index.csv / no class folders)
    # Heuristic fallback: *_binarized* -> 1, others -> 0.
    flat_tifs = [
        f.resolve()
        for f in data_root.glob("*")
        if f.is_file() and f.suffix.lower() in {".tif", ".tiff"}
    ]
    if flat_tifs:
        print("Warning: dataset_index.csv/class folders not found. Using heuristic labels from filename: 'binarized' => 1, otherwise 0.")
        for f in flat_tifs:
            name = f.name.lower()
            label = 1 if "binarized" in name else 0
            rows.append({"path": f, "label": label})

    return pd.DataFrame(rows)


def _extract_feature_matrix(df: pd.DataFrame, target_size: int = 64) -> np.ndarray:
    """Extract features with best available path:
    1) Frozen CNN embeddings if notebook helpers exist.
    2) Fallback handcrafted features from raw image statistics.
    """
    # Preferred path: reuse notebook's CNN embedding helper
    if all(k in globals() for k in ["OCTTifDataset", "DataLoader", "BATCH_SIZE", "NUM_WORKERS", "frozen_feature_extractor", "cnn_embeddings_from_backbone", "DEVICE", "eval_transform"]):
        try:
            ds = OCTTifDataset(df, eval_transform)
            dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
            backbone = frozen_feature_extractor().to(DEVICE)
            return cnn_embeddings_from_backbone(backbone, dl, DEVICE)
        except Exception as e:
            print("CNN embedding path failed, using fallback features:", e)

    # Fallback path: robust handcrafted features
    feats = []
    for p in df["path"].tolist():
        arr = None
        if "load_tif_as_array" in globals():
            arr = load_tif_as_array(Path(p))
        else:
            try:
                import tifffile as tiff
                arr = tiff.imread(str(p)).astype(np.float32)
                if arr.ndim == 3:
                    arr = arr[..., 0] if arr.shape[-1] == 1 else arr.mean(axis=-1)
                arr_min, arr_max = float(arr.min()), float(arr.max())
                if arr_max > arr_min:
                    arr = (arr - arr_min) / (arr_max - arr_min)
            except Exception:
                from PIL import Image
                arr = np.array(Image.open(p), dtype=np.float32)
                if arr.ndim == 3:
                    arr = arr[..., 0] if arr.shape[-1] == 1 else arr.mean(axis=-1)
                arr_min, arr_max = float(arr.min()), float(arr.max())
                if arr_max > arr_min:
                    arr = (arr - arr_min) / (arr_max - arr_min)

        h, w = arr.shape
        yy = np.linspace(0, h - 1, target_size).astype(int)
        xx = np.linspace(0, w - 1, target_size).astype(int)
        small = arr[np.ix_(yy, xx)]

        # features: downsampled pixels + summary stats
        summary = np.array([
            float(arr.mean()),
            float(arr.std()),
            float(np.percentile(arr, 10)),
            float(np.percentile(arr, 50)),
            float(np.percentile(arr, 90)),
            float(arr[:, : w // 2].mean()),
            float(arr[:, w // 2 :].mean()),
        ], dtype=np.float32)

        feats.append(np.concatenate([small.flatten().astype(np.float32), summary], axis=0))

    return np.vstack(feats)


def run_additional_ml_models(df: pd.DataFrame, n_folds: int = 5):
    """Evaluate models once on full dataset (no scenarios)."""
    if df is None or len(df) < 2:
        raise ValueError("Need at least 2 samples to train models.")
    if df["label"].astype(int).nunique() < 2:
        raise ValueError("Need both classes (0 and 1).")

    y = df["label"].astype(int).values
    min_class = int(pd.Series(y).value_counts().min())
    use_folds = min(n_folds, min_class)
    if use_folds < 2:
        raise ValueError("Not enough samples per class for stratified cross-validation.")

    X = _extract_feature_matrix(df)
    skf = StratifiedKFold(n_splits=use_folds, shuffle=True, random_state=42)
    split_indices = list(skf.split(X, y))

    models_map = {
        "logistic_regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
        ]),
        "knn": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=7, weights="distance")),
        ]),
        "random_forest": RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        ),
        "svm_rbf": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=2.0, gamma="scale", probability=True, class_weight="balanced", random_state=42)),
        ]),
    }

    try:
        import xgboost as xgb
        models_map["xgboost"] = xgb.XGBClassifier(
            n_estimators=250,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            eval_metric="logloss",
        )
    except Exception:
        pass

    rows = []
    for model_name, clf in models_map.items():
        oof_prob = np.zeros(len(df), dtype=np.float32)
        fold_acc = []

        for tr, va in split_indices:
            X_tr, X_va = X[tr], X[va]
            y_tr, y_va = y[tr], y[va]

            fold_clf = clone(clf)
            if model_name == "knn":
                k = max(1, min(7, len(y_tr)))
                fold_clf.set_params(clf__n_neighbors=k)

            fold_clf.fit(X_tr, y_tr)

            if hasattr(fold_clf, "predict_proba"):
                proba = fold_clf.predict_proba(X_va)[:, 1]
            elif hasattr(fold_clf, "decision_function"):
                score = fold_clf.decision_function(X_va)
                proba = 1.0 / (1.0 + np.exp(-score))
            else:
                proba = fold_clf.predict(X_va).astype(np.float32)

            oof_prob[va] = proba
            fold_acc.append(_compute_metrics_local(y_va, proba)["accuracy"])

        m = _compute_metrics_local(y, oof_prob)
        rows.append(
            {
                "model": model_name,
                "cv_accuracy": float(np.mean(fold_acc)),
                "accuracy": m["accuracy"],
                "sensitivity": m["sensitivity"],
                "specificity": m["specificity"],
                "precision": m["precision"],
                "f1": m["f1"],
                "auc": m["auc"],
            }
        )

    return pd.DataFrame(rows)[["model", "cv_accuracy", "accuracy", "sensitivity", "specificity", "precision", "f1", "auc"]]


# ------- Run block -------
PROJECT_ROOT = Path.cwd()
DATA_ROOT = globals().get("DATA_ROOT", Path(os.environ.get("OCT_DATA_ROOT", str(PROJECT_ROOT))))
df_all = globals().get("df_all", None)

def _bootstrap_labels_csv_if_missing(data_root: Path):
    """Create a starter dataset_index.csv when flat TIFF files exist but labels are absent."""
    csv_path = data_root / "dataset_index.csv"
    if csv_path.exists():
        return

    flat_tifs = sorted(
        [f for f in data_root.glob("*") if f.is_file() and f.suffix.lower() in {".tif", ".tiff"}],
        key=lambda p: p.name.lower(),
    )
    if not flat_tifs:
        return

    rows = []
    for f in flat_tifs:
        # Bootstrap only; user can edit dataset_index.csv later with true clinical labels.
        label = 1 if "binarized" in f.name.lower() else 0
        rows.append({"path": f.name, "label": label})

    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"Created starter labels file: {csv_path}")


# Build df_all robustly (no dependency on earlier cells execution order)
if df_all is None or len(df_all) == 0:
    # If no labels/class folders exist but flat TIFFs do, create a starter dataset_index.csv.
    _bootstrap_labels_csv_if_missing(DATA_ROOT)

    if "collect_samples" in globals():
        try:
            df_all = collect_samples(DATA_ROOT)
        except Exception:
            df_all = _safe_collect_samples(DATA_ROOT)
    else:
        df_all = _safe_collect_samples(DATA_ROOT)

# Strict mode: real data only (no synthetic generation)
if df_all is None or len(df_all) == 0:
    raise ValueError(
        f"No data found in {DATA_ROOT}. Put real .tif/.tiff files there (or set OCT_DATA_ROOT), and provide dataset_index.csv or class folders."
    )

# Remove synthetic samples if present
path_s = df_all["path"].astype(str)
real_mask = ~path_s.str.contains("syn_", case=False, regex=False)
if real_mask.sum() < len(df_all):
    removed = len(df_all) - int(real_mask.sum())
    print(f"Removed {removed} synthetic samples (paths containing 'syn_').")

df_all = df_all.loc[real_mask].reset_index(drop=True)

# Keep only TIFF/TIFF
ext_mask = df_all["path"].astype(str).str.lower().str.endswith((".tif", ".tiff"))
df_all = df_all.loc[ext_mask].reset_index(drop=True)

if len(df_all) < 2:
    raise ValueError(
        "Not enough real labeled .tif/.tiff data after filtering. Add actual OCT files and dataset_index.csv."
    )

if "label" not in df_all.columns:
    raise ValueError("Missing 'label' column. Provide dataset_index.csv with columns: path,label")

if df_all["label"].nunique() < 2:
    raise ValueError("Dataset must contain both classes (0 and 1). Edit dataset_index.csv and rerun.")

print(f"Using {len(df_all)} real TIFF samples from: {DATA_ROOT}")
print(df_all[["path", "label"]].head())

# Paper-reported benchmarks (Hoyek et al., RETINA 2026) for quick comparison
paper_metrics = pd.DataFrame(
    [
        {
            "scenario": "No duplication, 2 scans/patient, KNN",
            "cv_accuracy": 0.7711,
            "accuracy": 1.00,
            "sensitivity": 1.00,
            "specificity": 1.00,
            "precision": 1.00,
            "f1": 1.00,
            "auc": np.nan,
        },
        {
            "scenario": "With duplication, 1 scan/patient, Random Forest",
            "cv_accuracy": 0.96,
            "accuracy": 1.00,
            "sensitivity": 1.00,
            "specificity": 1.00,
            "precision": 1.00,
            "f1": 1.00,
            "auc": 1.00,
        },
        {
            "scenario": "With duplication, 2 scans/patient, Random Forest",
            "cv_accuracy": 0.8958,
            "accuracy": 0.90,
            "sensitivity": 1.00,
            "specificity": 0.89,
            "precision": 0.33,
            "f1": 0.50,
            "auc": 0.89,
        },
        {
            "scenario": "With duplication, 3 scans/patient, Random Forest",
            "cv_accuracy": 0.9423,
            "accuracy": 0.8571,
            "sensitivity": 1.00,
            "specificity": 0.76,
            "precision": 0.73,
            "f1": 0.85,
            "auc": 0.90,
        },
    ]
)
print("\nPaper benchmark metrics:")
display(paper_metrics)

effective_folds = int(globals().get("N_FOLDS", 5))
ml_compare_df = run_additional_ml_models(df_all, n_folds=effective_folds)
print("\nYour model metrics (paper metrics only):")
display(ml_compare_df)
ml_compare_df

Using 8 real TIFF samples from: /Users/tripa/Desktop/Projects/Hospital
                                                path  label
0  /Users/tripa/Desktop/Projects/Hospital/19_OD_D...      0
1  /Users/tripa/Desktop/Projects/Hospital/19_OD_D...      1
2  /Users/tripa/Desktop/Projects/Hospital/19_OD_S...      0
3  /Users/tripa/Desktop/Projects/Hospital/19_OD_S...      1
4  /Users/tripa/Desktop/Projects/Hospital/19_OS_D...      0

Paper benchmark metrics:


,scenario,cv_accuracy,accuracy,sensitivity,specificity,precision,f1,auc
0,"No duplication, 2 scans/patient, KNN",0.7711,1.0000,1.0,1.00,1.00,1.00,NaN
1,"With duplication, 1 scan/patient, Random Forest",0.9600,1.0000,1.0,1.00,1.00,1.00,1.00
2,"With duplication, 2 scans/patient, Random Forest",0.8958,0.9000,1.0,0.89,0.33,0.50,0.89
3,"With duplication, 3 scans/patient, Random Forest",0.9423,0.8571,1.0,0.76,0.73,0.85,0.90



Your model metrics (paper metrics only):


,model,cv_accuracy,accuracy,sensitivity,specificity,precision,f1,auc
0,logistic_regression,0.500,0.500,0.0,1.00,0.000000,0.000000,0.8125
1,knn,0.500,0.500,0.0,1.00,0.000000,0.000000,0.9375
2,random_forest,1.000,1.000,1.0,1.00,1.000000,1.000000,1.0000
3,svm_rbf,0.625,0.625,1.0,0.25,0.571429,0.727273,0.7500
4,xgboost,0.500,0.500,0.0,1.00,0.000000,0.000000,0.5000


,model,cv_accuracy,accuracy,sensitivity,specificity,precision,f1,auc
0,logistic_regression,0.500,0.500,0.0,1.00,0.000000,0.000000,0.8125
1,knn,0.500,0.500,0.0,1.00,0.000000,0.000000,0.9375
2,random_forest,1.000,1.000,1.0,1.00,1.000000,1.000000,1.0000
3,svm_rbf,0.625,0.625,1.0,0.25,0.571429,0.727273,0.7500
4,xgboost,0.500,0.500,0.0,1.00,0.000000,0.000000,0.5000
